# Forschungsfrage 4 - Semantische Heterogenität: Moderner Non-LLM-Ansatz (Sentence Transformers)

`thread_category` + `title` werden mit einem vortrainierten Sentence-Transformer-Modell
(`sentence-transformers/all-MiniLM-L6-v2`, 384-dimensionale semantische Einbettungen)
vektorisiert; derselbe Random-Forest-Klassifikator wie im vorherigen TF-IDF-Ansatz
ordnet jede Zeile einer der 12 kanonischen `parent_category`-Klassen zu. Der
Klassifikator wird bewusst **identisch** zur ursprünglichen TF-IDF-Variante gehalten,
damit ausschließlich **eine** Variable verglichen wird: klassische Bag-of-Words-
Repräsentation (TF-IDF) vs. vortrainierte semantische Satz-Einbettung.

**Methodische Anmerkung zur Ausführungsumgebung:** Der Download vortrainierter
Modelle von huggingface.co ist im Cloud-Sandbox dieser Arbeit durch eine
Organisations-Firewall blockiert (verifiziert über direkten Verbindungstest).
Die Einbettungen wurden daher in einem separaten Schritt lokal auf einem Rechner
mit Internetzugriff berechnet (`build_tf4_embeddings_generator_script.py`, ohne
API-Key, ohne Kosten - rein lokales Open-Source-Modell) und als `.npy`-Dateien in
diese Umgebung importiert. Dieses Notebook selbst benötigt keinen Internetzugriff.

Bewertung auf demselben `test`-Split wie im TF-IDF+RF- und im Dictionary/Fuzzy-Notebook.


## 1. Setup: vorab berechnete Einbettungen laden

In [1]:
import pandas as pd
import numpy as np
import json
import time
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42
os.makedirs("results", exist_ok=True)

eval_embeddings = np.load("results/tf4_sentence_embeddings_eval.npy")
app_embeddings = np.load("results/tf4_sentence_embeddings_app.npy")
eval_meta = pd.read_csv("results/tf4_sentence_embeddings_eval_meta.csv")
app_meta = pd.read_csv("results/tf4_sentence_embeddings_app_meta.csv")

with open("results/tf4_sentence_embeddings_info.json") as f:
    emb_info = json.load(f)

print(f"Modell: {emb_info['model_name']}  Dimension: {emb_info['embedding_dim']}")
print(f"Eval-Einbettungen: {eval_embeddings.shape}  Anwendungs-Einbettungen: {app_embeddings.shape}")

assert len(eval_meta) == eval_embeddings.shape[0]
assert len(app_meta) == app_embeddings.shape[0]

train_mask = (eval_meta["split"] == "train").values
test_mask = (eval_meta["split"] == "test").values


Modell: sentence-transformers/all-MiniLM-L6-v2  Dimension: 384
Eval-Einbettungen: (825, 384)  Anwendungs-Einbettungen: (500, 384)


## 2. Training und Bewertung

In [2]:
X_train, y_train = eval_embeddings[train_mask], eval_meta.loc[train_mask, "true_parent_category"]
X_test, y_test = eval_embeddings[test_mask], eval_meta.loc[test_mask, "true_parent_category"]

t0 = time.time()
clf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1, class_weight="balanced")
clf.fit(X_train, y_train)
train_time = time.time() - t0

y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {accuracy:.3f}  Macro-F1: {macro_f1:.3f}  (n_test={len(y_test)}, Trainingszeit {train_time:.2f}s)")
print()
print(classification_report(y_test, y_pred))


Accuracy: 0.710  Macro-F1: 0.448  (n_test=124, Trainingszeit 1.65s)

                         precision    recall  f1-score   support

                Apparel       1.00      0.75      0.86         4
             Automotive       1.00      0.20      0.33         5
      Beauty & Wellness       1.00      0.60      0.75         5
Computers & Electronics       0.68      0.98      0.81        51
          Entertainment       1.00      0.33      0.50         6
     Financial Services       1.00      0.40      0.57         5
          Home & Garden       0.65      0.77      0.71        31
          Kids & Babies       0.00      0.00      0.00         4
            Restaurants       1.00      0.75      0.86         4
         Small Business       0.00      0.00      0.00         2
       Sports & Fitness       0.00      0.00      0.00         5
                 Travel       0.00      0.00      0.00         2

               accuracy                           0.71       124
              macro

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## 3. Ergebnisse speichern (Test-Bewertung + Anwendungsmenge)

In [3]:
test_out = eval_meta.loc[test_mask, ["row_id", "true_parent_category"]].copy()
test_out["parent_category_pred_st"] = y_pred
test_out.to_csv("results/tf4_sentencetransformers_predictions.csv", index=False)

app_pred = clf.predict(app_embeddings)
app_bench = pd.read_csv("benchmark/tf4_application_set.csv")
app_out = app_bench[["row_id", "thread_category", "title"]].copy()
app_out["parent_category_pred_st"] = app_pred
app_out.to_csv("results/tf4_sentencetransformers_application_predictions.csv", index=False)

metrics = {
    "experiment": "TF4_Semantik", "method": "SentenceTransformers_RandomForest",
    "embedding_model": emb_info["model_name"],
    "accuracy": accuracy, "macro_f1": macro_f1, "n_test": len(y_test),
    "train_time_sec": train_time, "n_application_rows": len(app_bench),
}
with open("results/tf4_sentencetransformers_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

log_rows = pd.DataFrame([{
    "experiment": "TF4_Semantik", "method": "SentenceTransformers_RandomForest", "n_items": len(y_test),
    "wall_time_sec": train_time, "input_tokens": 0, "output_tokens": 0,
    "estimated_cost_usd": 0.0, "model_name": "all-MiniLM-L6-v2+random_forest (kein LLM/API, lokales Open-Source-Modell)",
}])
log_path = "results/laufzeit_kosten_log.csv"
if os.path.exists(log_path):
    _old_log = pd.read_csv(log_path)
    _new_keys = set(zip(log_rows["experiment"], log_rows["method"]))
    _old_log = _old_log[~_old_log.apply(lambda r: (r["experiment"], r["method"]) in _new_keys, axis=1)]
    _combined_log = pd.concat([_old_log, log_rows], ignore_index=True)
else:
    _combined_log = log_rows
_combined_log.to_csv(log_path, index=False)

print("Gespeichert: results/tf4_sentencetransformers_predictions.csv, results/tf4_sentencetransformers_application_predictions.csv, results/tf4_sentencetransformers_metrics.json")


Gespeichert: results/tf4_sentencetransformers_predictions.csv, results/tf4_sentencetransformers_application_predictions.csv, results/tf4_sentencetransformers_metrics.json
